# S3.2 — verifier backbone ablation (7 arms × 5 seeds × 2 LRs)

**Paste-and-run on Kaggle.** One code cell.

Before running: **Internet ON**, **GPU T4 ON**, and attach `bn_clean.csv`
with *+ Add Input*.

⚠️ **Read `docs/protocol.md` §"S3.2 pre-commitment" BEFORE reading the
output.** The arms, the seed count, the decision rule and the tie-break were
all fixed on 2026-08-08, before any backbone was downloaded. Reading the
numbers first is how a pre-registration quietly becomes a post-hoc story.

⚠️ **A `TIE` is a pre-registered outcome, not a failed run.** The 2025–26
Bangla literature reports three different winners on the same dataset, so a
tie is the honest and expected result. Do not re-run with different settings
to break one.

**Budget:** 7 arms × 2 LRs × 5 seeds = **70 fine-tuning runs** on 804 rows of
~8-word text. Short, but not free — expect a few hours on a T4, and Kaggle's
session limit is 12h. If it will not fit, run it in two sessions **by arm**
(never by seed), and say so in the lab notebook.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  S3.2 on Kaggle — paste as ONE cell and run.
#
#  This notebook is a RUNNER. It clones, installs, checks, and calls the
#  script. No logic lives here: anything computed in a notebook cell cannot
#  enter the paper (CLAUDE.md, working conventions).
# ══════════════════════════════════════════════════════════════════════════

%cd /kaggle/working
!rm -rf /kaggle/working/thesis
!git clone --depth 1 https://github.com/alphapie77/BSc_Thesis.git /kaggle/working/thesis
%cd /kaggle/working/thesis
!git log --oneline -1

!pip install -q transformers datasets setfit pyyaml scikit-learn

import shutil
from pathlib import Path

hits = sorted(Path('/kaggle/input').rglob('bn_clean.csv'))
if not hits:
    visible = [str(p) for p in Path('/kaggle/input').rglob('*') if p.is_file()][:20]
    raise FileNotFoundError(
        "bn_clean.csv is not under /kaggle/input. Attach it with '+ Add Input'. "
        f"Visible now: {visible or 'NOTHING'}"
    )
Path('data/cleaned').mkdir(parents=True, exist_ok=True)
shutil.copy(hits[0], 'data/cleaned/bn_clean.csv')
print('input:', hits[0])

# ── Gate 1: the contract tests. They need no GPU and no torch. If the split
# ── contract is broken, everything after this is contaminated, so stop here.
!python -m pytest tests/test_s3_backbone.py -q

# ── Gate 2: the dry run. Proves the plumbing end-to-end and re-checks that
# ── n is still 804/82 on THIS host, before any weights are downloaded.
!python -m src.verifier.s3_backbone_ablation --config configs/s3_backbone.yaml --dry-run

# ── The environment these numbers belong to. Mandatory on any non-local host
# ── (provenance fact (env)): requirements.lock.txt is Windows-frozen and does
# ── NOT describe this run.
!python -m src.common.env_snapshot --out results/env_snapshot_s3_kaggle.json

# ── The real run. 70 fine-tuning runs; this is the long part.
!python -m src.verifier.s3_backbone_ablation --config configs/s3_backbone.yaml

# ── Package for download, so results are committed WITH their notebook entry.
import zipfile
OUT = Path('/kaggle/working/s3_backbone_outputs.zip')
wanted = [
    'results/s3_backbone_ablation.md',
    'results/s3_backbone_ablation.json',
    'results/s3_backbone_per_seed.csv',
    'results/env_snapshot_s3_kaggle.json',
]
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in wanted:
        if Path(f).exists():
            z.write(f)
        else:
            print('MISSING:', f)
print('wrote', OUT)

print(open('results/s3_backbone_ablation.md', encoding='utf-8').read())